valid_x,yを用いて、transactionから候補リストを作成する

In [1]:
import pandas as pd
from pathlib import Path

# どの環境でも動くパス設定
DATA_DIR = (Path.cwd() / '../../data').resolve()

dtype_config = {
    "article_id": "string",
    "week": "int8",
    "price": "float32",
    "sales_channel_id": "int8"
}

transactions = pd.read_csv(DATA_DIR / 'processed/transactions_with_week.csv', dtype=dtype_config)
valid_x = pd.read_csv(DATA_DIR / 'processed/valid_x.csv', dtype=dtype_config)
valid_y = pd.read_csv(DATA_DIR / 'processed/valid_y.csv', dtype=dtype_config)

print("読み込み完了")
print(f"transactions: {len(transactions)}行")
print(f"valid_x: {len(valid_x)}行")
print(f"valid_y: {len(valid_y)}行")

読み込み完了
transactions: 31788324行
valid_x: 31575041行
valid_y: 213283行


In [2]:
valid_x.head()


,t_dat,customer_id,article_id,price,sales_channel_id,week
0,2018-09-20,3,0663713001,0.050831,2,0
1,2018-09-20,3,0541518023,0.030492,2,0
2,2018-09-20,8,0505221004,0.015237,2,0
3,2018-09-20,8,0685687003,0.016932,2,0
4,2018-09-20,8,0685687004,0.016932,2,0


先週の上位30件を候補にする

In [4]:
last_week = valid_x["week"].max()
print(f"valid_xの最終週: {last_week}")

top50 = (
    valid_x[valid_x["week"] == last_week]
    .groupby("article_id")["customer_id"]
    
    .count()
    .reset_index()
    .rename(columns={"customer_id": "count"})
    .sort_values("count", ascending=False)
    .head(50)
)

print(top50)

valid_xの最終週: 103
       article_id  count
16715  0909370001   1310
17692  0924243001    817
12136  0865799006    750
17368  0918522001    666
448    0448509014    577
4728   0751471001    576
17674  0923758001    548
7561   0809238001    539
5267   0762846027    538
17354  0918292001    527
7562   0809238005    490
17369  0918525001    433
17958  0929165002    431
17189  0915529003    430
17693  0924243002    430
2471   0673677002    420
17186  0915526001    420
17190  0915529005    408
16865  0910601003    408
3319   0706016001    408
10524  0850917001    407
15545  0896169005    391
11874  0863583001    391
3627   0715624001    390
17428  0919273002    385
16716  0909371001    383
7263   0805947001    382
4744   0751471043    379
11899  0863646001    376
17120  0914805002    355
3320   0706016002    335
3598   0714790020    334
3321   0706016003    330
31     0158340001    329
11888  0863595006    328
10694  0852584001    323
16664  0908799002    320
15432  0894780001    318
15776  0

In [ ]:
# valid_xに登場する全ユーザーを取得
all_users = valid_x["customer_id"].unique()
print(f"ユーザー数: {len(all_users)}")

# 全ユーザー × top30商品の候補ペアを作成
candidate_top30 = pd.DataFrame({
    "customer_id": all_users.repeat(50),
    "article_id": list(top50["article_id"].values) * len(all_users)
})

print(f"候補ペア数: {len(candidate_top30)}")
print(candidate_top30.head(10))

ユーザー数: 1357359
候補ペア数: 67867950
   customer_id  article_id
0            3  0909370001
1            3  0924243001
2            3  0865799006
3            3  0918522001
4            3  0448509014
5            3  0751471001
6            3  0923758001
7            3  0809238001
8            3  0762846027
9            3  0918292001


In [5]:
print(f"ユーザー数: {len(all_users)}")
print(f"作成される行数: {len(all_users) * 30:,}")

ユーザー数: 1357359
作成される行数: 40,720,770


In [8]:
# valid_yと照合してRecallを計算する
# valid_yは「実際に買われた（ユーザー, 商品）ペア」= 正解

# 候補ペアと正解ペアをマージして一致するものを探す
hit = candidate_top30.merge(
    valid_y[["customer_id", "article_id"]],
    on=["customer_id", "article_id"],
    how="inner"
)

# Recall = 正解のうち候補に含まれた割合
recall = len(hit) / len(valid_y)
print(f"正解数: {len(valid_y)}")
print(f"候補に含まれた正解数: {len(hit)}")
print(f"Recall: {recall:.4f}")

正解数: 213283
候補に含まれた正解数: 14322
Recall: 0.0672
